In [1]:
import pandas as pd
import numpy as np
import boto3
from io import BytesIO
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
)
BUCKET = "oil-pipeline"

def read_parquet(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))

def save_parquet(df, key):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
    print(f"Saved: {key} ({len(df)} rows)")

In [2]:
production = read_parquet("cleaned/production/data.parquet")
targets = read_parquet("cleaned/well_targets/well_targets.parquet")

production["date"] = pd.to_datetime(production["date"])
targets["date"] = pd.to_datetime(targets["date"])

print(production.shape, targets.shape)
production.head()

(120, 10) (90, 3)


,prod_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4
1,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0
2,3,1,2025-10-03,211.9,55040.0,183.2,7430.0,0.7,88.5,119.8
3,4,1,2025-10-04,215.1,55500.0,180.5,7515.0,0.2,87.6,121.5
4,5,1,2025-10-05,214.6,55380.0,182.0,7480.0,0.4,88.0,120.9


In [3]:
df = production.merge(targets, on=["well_id", "date"])

print(df.shape)
df.head()

(90, 11)


,prod_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure,daily_oil_ton
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4,212.4
1,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0,213.8
2,3,1,2025-10-03,211.9,55040.0,183.2,7430.0,0.7,88.5,119.8,211.9
3,4,1,2025-10-04,215.1,55500.0,180.5,7515.0,0.2,87.6,121.5,215.1
4,5,1,2025-10-05,214.6,55380.0,182.0,7480.0,0.4,88.0,120.9,214.6


In [4]:
features = [
    "pressure", "temperature",
    "energy_kwh", "downtime_hours",
    "gas_m3", "water_m3"
]
target = "daily_oil_ton"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 72, Test: 18


In [5]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = mean_squared_error(y_test, y_pred_lr, squared=False)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = mean_squared_error(y_test, y_pred_rf, squared=False)

print(f"Linear Regression - MAE: {mae_lr:.3f}, RMSE: {rmse_lr:.3f}")
print(f"Random Forest     - MAE: {mae_rf:.3f}, RMSE: {rmse_rf:.3f}")

Linear Regression - MAE: 0.217, RMSE: 0.277
Random Forest     - MAE: 0.262, RMSE: 0.343


In [6]:
results = X_test.copy()
results["well_id"] = df.loc[X_test.index, "well_id"].values
results["date"] = df.loc[X_test.index, "date"].values
results["actual"] = y_test.values
results["predicted_lr"] = y_pred_lr
results["predicted_rf"] = y_pred_rf
results["error_lr"] = results["actual"] - results["predicted_lr"]
results["error_rf"] = results["actual"] - results["predicted_rf"]

print(results[["well_id", "date", "actual", "predicted_rf", "error_rf"]])
save_parquet(results, "marts/mart_ml_predictions.parquet")

    well_id       date  actual  predicted_rf  error_rf
40        2 2025-10-11   184.7    184.848000 -0.148000
22        1 2025-10-23   212.7    212.474000  0.226000
55        2 2025-10-26   184.5    184.641000 -0.141000
70        5 2025-10-11   197.9    197.806583  0.093417
0         1 2025-10-01   212.4    212.919000 -0.519000
26        1 2025-10-27   213.9    213.809000  0.091000
39        2 2025-10-10   185.9    185.785000  0.115000
65        5 2025-10-06   199.1    199.204267 -0.104267
10        1 2025-10-11   211.4    211.940000 -0.540000
44        2 2025-10-15   185.7    185.738000 -0.038000
81        5 2025-10-22   198.7    198.623000  0.077000
35        2 2025-10-06   185.4    185.133000  0.267000
56        2 2025-10-27   185.1    185.255000 -0.155000
86        5 2025-10-27   198.0    197.976564  0.023436
12        1 2025-10-13   214.9    214.167000  0.733000
4         1 2025-10-05   214.6    213.894000  0.706000
18        1 2025-10-19   214.0    214.271000 -0.271000
28        